# Train YOLOv8 on Retail Products Classification

GPU half of the VS Code workflow. You edit the repo locally, push, and this notebook pulls
that commit and trains on Colab's GPU. Nothing is edited here — cell 3 hard-resets to origin.

**Run order:** 1 → 8 top to bottom. Cell 7 is a cheap smoke run; only go to cell 8 once it passes.

The dataset labels whole images across 21 categories, so this trains `yolov8n-cls`
(classification). Pass `--task detect` instead to train the full-frame-box detector shim.

## 1. Check the GPU
If this fails: **Runtime → Change runtime type → T4 GPU**, then re-run.

In [ ]:
import subprocess

import torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip() or "nvidia-smi returned nothing")

assert torch.cuda.is_available(), "No GPU attached. Runtime > Change runtime type > T4 GPU."
print(f"torch {torch.__version__} | cuda {torch.version.cuda}")

## 2. Mount Drive
Colab wipes `/content` when the runtime recycles, so weights and the Kaggle token live in Drive.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/yolo-retail"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("artifacts will be saved to", DRIVE_DIR)

## 3. Pull the repo

This is the bridge: whatever you last pushed from VS Code is what trains here.

`git reset --hard` **discards any edit made inside Colab** — that is deliberate, so the
notebook can never train a version that does not exist in git. Edit in VS Code, push, re-run.

Private repo? Swap `REPO_URL` for `https://<token>@github.com/weshallsah/yolo.git` using a
fine-grained PAT with read-only Contents access.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/weshallsah/yolo.git"
BRANCH = "main"
REPO_DIR = "/content/yolo"


def sh(*args, cwd=None):
    print("$", " ".join(args))
    subprocess.run(args, cwd=cwd, check=True)


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "fetch", "origin", BRANCH, cwd=REPO_DIR)
    sh("git", "checkout", BRANCH, cwd=REPO_DIR)
    sh("git", "reset", "--hard", f"origin/{BRANCH}", cwd=REPO_DIR)
else:
    sh("git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
sh("git", "log", "--oneline", "-1")

## 4. Install dependencies
Only what training needs — not the FastAPI serving stack.

In [ ]:
!pip install -q "ultralytics==8.4.150" kaggle

import ultralytics

ultralytics.checks()

## 5. Kaggle credentials

Get your token at **kaggle.com → Settings → API → Create New Token**, which downloads
`kaggle.json` containing a `username` and a `key`.

Then pick one (the first is better — it survives runtime recycles and puts nothing in Drive):

- **Colab Secrets** — key icon in the left sidebar → *Add new secret*. Add `KAGGLE_USERNAME`
  and `KAGGLE_KEY` with the two values from that file, and toggle **Notebook access** on.
- **Drive** — put `kaggle.json` at `MyDrive/kaggle/kaggle.json`.

This cell never prompts and never blocks: if it finds neither, it stops with instructions.
The key is written to disk for the CLI but is never printed.

In [ ]:
import json
import os

TARGETS = ["/root/.kaggle/kaggle.json", "/root/.config/kaggle/kaggle.json"]
DRIVE_TOKEN = "/content/drive/MyDrive/kaggle/kaggle.json"

token = None

# Colab Secrets first: no widget, no prompt, and it outlives the runtime. userdata.get raises
# if the secret is missing or notebook access is off, so both fall through to Drive.
try:
    from google.colab import userdata

    token = {"username": userdata.get("KAGGLE_USERNAME"), "key": userdata.get("KAGGLE_KEY")}
    print("using Colab Secrets")
except Exception as exc:
    print(f"Colab Secrets unavailable ({type(exc).__name__}) - falling back to Drive")

if token is None and os.path.exists(DRIVE_TOKEN):
    with open(DRIVE_TOKEN) as handle:
        token = json.load(handle)
    print("using the kaggle.json in Drive")

if token is None or not token.get("username") or not token.get("key"):
    raise SystemExit(
        "No Kaggle credentials found. Either:\n"
        "  A. Colab Secrets (recommended) - key icon in the left sidebar > Add new secret.\n"
        "     Add KAGGLE_USERNAME and KAGGLE_KEY from kaggle.json, turn on Notebook access,\n"
        "     then re-run this cell.\n"
        f"  B. Put kaggle.json in Drive at {DRIVE_TOKEN} and re-run this cell.\n"
        "Get the file from kaggle.com > Settings > API > Create New Token."
    )

# Written to both paths because which one the CLI reads depends on its version.
for target in TARGETS:
    os.makedirs(os.path.dirname(target), exist_ok=True)
    with open(target, "w") as handle:
        json.dump(token, handle)
    os.chmod(target, 0o600)

print("credentials installed for Kaggle user:", token["username"])

## 6. Download the competition data

The cell probes your credentials against a public endpoint first, so that when the
competition download fails you know which of two unrelated things went wrong:

- the key itself is bad → regenerate the token
- the key is fine but the competition will not serve you → accept the rules, or you are not
  enrolled in what is an InClass competition

Kaggle answers **401** for unaccepted rules on competition downloads (not 403, which is what
you would expect), so the status code alone cannot tell those apart — hence the probe.

Archives unpack into `/content/retail_data`, one of the roots `prepare_retail_dataset.py`
searches, so no `--csv` flag is needed afterwards.

In [ ]:
import glob
import os
import subprocess
import zipfile

COMPETITION = "retail-products-classification"
DATA_DIR = "/content/retail_data"
os.makedirs(DATA_DIR, exist_ok=True)


def kaggle(*args):
    return subprocess.run(["kaggle", *args], capture_output=True, text=True)


# Probe: a public listing needs valid credentials but no competition membership, so it
# separates "key is wrong" from "key is fine, competition says no".
probe = kaggle("datasets", "list", "-s", "retail", "--page-size", "1")
credentials_ok = probe.returncode == 0
print("credentials:", "OK" if credentials_ok else "REJECTED")
if not credentials_ok:
    print(probe.stdout, probe.stderr)
    raise SystemExit(
        "Kaggle rejected your credentials, so the competition is not the problem yet.\n"
        "A key is exactly 32 hex characters: check KAGGLE_USERNAME is your username (not "
        "your email) and KAGGLE_KEY is the key alone, with no trailing whitespace and not "
        "the whole kaggle.json. Creating a new token revokes the previous one, so if you "
        "clicked 'Create New Token' more than once only the newest key works."
    )

result = kaggle("competitions", "download", "-c", COMPETITION, "-p", DATA_DIR)
print(result.stdout, result.stderr)
if result.returncode != 0:
    raise SystemExit(
        f"Your credentials work, so this is about access to '{COMPETITION}' specifically.\n"
        f"1. Sign in as the same account and open "
        f"https://www.kaggle.com/c/{COMPETITION}/rules, then click 'I Understand and "
        "Accept'. Kaggle returns 401 (not 403) until you do.\n"
        "2. If that page offers no accept button, or the Data tab is unreachable in the "
        "browser, this InClass competition is closed or restricted to enrolled students "
        "and no token will open it. Point the pipeline at another source instead:\n"
        "   python backend/scripts/train_on_retail.py --task classify \\\n"
        "       --csv <train.csv> --images-dir <images/>"
    )

# Two passes: the outer archive, then any zips nested inside it (train.csv.zip, image zips).
# Each archive is deleted once extracted, since Colab's disk is smaller than you think.
for _ in range(2):
    for archive in glob.glob(os.path.join(DATA_DIR, "**", "*.zip"), recursive=True):
        print("extracting", archive)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(os.path.dirname(archive))
        os.remove(archive)

print(subprocess.run(["du", "-sh", DATA_DIR], capture_output=True, text=True).stdout)
for entry in sorted(glob.glob(os.path.join(DATA_DIR, "*")))[:20]:
    print(" ", entry)

## 7. Smoke run
Builds a 2,000-image subset and trains one epoch. Catches a bad download, a column-name
surprise or an OOM in ~2 minutes instead of an hour into the real run.

In [ ]:
!python backend/scripts/train_on_retail.py \
    --task classify \
    --max-images 2000 \
    --epochs 1 \
    --batch 64 \
    --workers 2 \
    --cache none

## 8. Full training run

`--force` is required: cell 7 left a capped 2,000-image dataset behind, and without it the
build is skipped and you would train on the subset again.

`--cache disk` because ~42k images will not fit in Colab's RAM. Watch `top-1` accuracy.
Raise `--epochs` if it is still climbing when it stops; add `--max-per-class 3000` if the
per-class counts printed by the prep step look badly skewed.

In [ ]:
!python backend/scripts/train_on_retail.py \
    --task classify \
    --force \
    --epochs 30 \
    --imgsz 224 \
    --batch 64 \
    --workers 2 \
    --cache disk

## 9. Save the run to Drive
Do this before the runtime recycles, or the weights are gone.

In [ ]:
import os
import shutil

RUN_DIR = "backend/models/retail_yolo_products_cls"
destination = os.path.join(DRIVE_DIR, "retail_yolo_products_cls")

shutil.copytree(RUN_DIR, destination, dirs_exist_ok=True)
print("saved to", destination)
!ls -la {destination}/weights

## 10. Sanity-check the trained classifier
Predicts a few held-out validation images. The printed class should usually match the
directory the image came from.

In [ ]:
import glob
import os

from ultralytics import YOLO

model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))
samples = sorted(glob.glob("backend/training_data/retail/dataset_products_cls/val/*/*"))[:10]

for result in model.predict(samples, verbose=False):
    truth = os.path.basename(os.path.dirname(result.path))
    predicted = result.names[result.probs.top1]
    mark = "ok " if truth == predicted else "MISS"
    print(f"{mark} true={truth:28s} pred={predicted:28s} conf={result.probs.top1conf:.2f}")

## Getting the weights back into VS Code

The run directory is in Drive, so download `best.pt` from
`MyDrive/yolo-retail/retail_yolo_products_cls/weights/` and drop it in the repo (it is
gitignored — `backend/models/` is not tracked), then point the backend at it:

```bash
# backend/.env
APP_YOLO_WEIGHTS_PATH=models/retail_yolo_products_cls/weights/best.pt
APP_MODEL_TASK=classify
```

`APP_MODEL_TASK=classify` is required. The default serving path reads `result.boxes`, which a
classifier does not produce — without it the API returns an empty list for every image.